# Tutorial - Three-Phase Induction Motor (TPIM) - Part 8
======================

## Section 4: Field-Oriented Control (FOC) Inverter

This notebook continues **tutorial_part_7.ipynb** (Section 3, `InverterVF`) and covers Section 4: driving the `MotorElement` with a three-phase voltage source inverter under indirect Field-Oriented Control (`InverterFOC`), through `.run_with_inverter_foc()`, followed by a comparison against the V/f drive.

This notebook is self-contained and can be run independently.

In [ ]:
import ross as rs
import numpy as np

from ross.units import Q_

# Make sure the default renderer is set to 'notebook' for inline plots in Jupyter
import plotly.io as pio
import plotly.graph_objects as go

pio.renderers.default = "notebook"

## 4.1 Instantiating the Motor

The motor parameters are again the same as in the previous notebooks (1.5 hp / 127 V / 60 Hz / 4 poles / 1710 RPM). As with `InverterVF`, `voltage_net` and `frequency_net` do not need to be informed at instantiation: `InverterFOC` internally regulates the voltage and frequency applied to the motor.

In [ ]:
motor4 = rs.MotorElement(
    n=0,
    tag="TPIM_FOC",
    power_nom=Q_(1.5, "hp"),
    voltage_nom=127,
    speed_nom=Q_(1710, "RPM"),
    frequency_nom=Q_(60.0, "Hz"),
    n_poles=4,
    stator_resistance=2.5,
    rotor_resistance=1.8,
    stator_reactance=1.3,
    rotor_reactance=1.3,
    mutual_reactance=43.08,
    Ip_motor=0.0372,
    viscosity_coeff=0.0,
    Ip_load=0.0,
)
motor4

## 4.2 Running with `run_with_inverter_foc`

The `.run_with_inverter_foc()` method simulates the motor driven by a three-phase voltage source inverter operating under indirect Field-Oriented Control (iFOC). Unlike `InverterVF`, which synthesizes voltages open-loop from a fixed V/f law, `InverterFOC` is a **closed-loop** element: at every simulation time step it reads back the instantaneous rotor speed and stator currents from the motor, and uses nested PI controllers (an outer speed loop plus inner d-axis/q-axis current loops, formulated in the rotor-flux reference frame) to regulate the shaft speed towards a reference derived from `frequency_ref`.

Besides the arguments already seen in `.run_with_inverter_vf()` (tutorial_part_7.ipynb), the relevant parameters are:

- `frequency_s`: IGBT switching frequency (`Fs`);
- `time_ramp`: acceleration ramp time for the synchronous frequency reference [s];
- `frequency_ref`: synchronous electrical frequency reference, analogous to `InverterVF`'s own `frequency_ref` (e.g. `Q_(60, "Hz")`). The closed speed loop converts it internally to the equivalent mechanical speed (`frequency_ref / (n_poles / 2)`) and drives the shaft towards it, correcting for slip. If `None`, the motor nominal frequency is used.

In this example, `frequency_ref` is set to the motor's nominal frequency (60 Hz), which corresponds to the nominal speed. Unlike the V/f case (where the steady-state speed settles below the open-loop reference because of the induction motor's natural slip), the closed speed loop is expected to keep the shaft speed close to the synchronous speed set by `frequency_ref`, even after the nominal load torque is applied at t = 1.5 s.

**Note:** because `InverterFOC` reads back the motor's instantaneous state at every simulation step, this loop cannot be pre-compiled the way the open-loop elements are, so it runs noticeably slower in real (wall-clock) time than `.run_with_inverter_vf()` for the same `t`/`time_step`. The same `time_step` guidance from tutorial_part_7.ipynb applies here too.

In [ ]:
Fs = 5000.0  # IGBT switching frequency [Hz], reused below for the FFT plots

dt = 1e-3
tf = 3.0
t4 = np.arange(0, tf + dt, dt)

results4 = motor4.run_with_inverter_foc(
    t4,
    time_step=1e-5,
    load_torque_entrance_time=1.5,
    load_torque_ratio=1.0,
    frequency_s=Q_(Fs, "Hz"),
    time_ramp=0.6667,
    frequency_ref=Q_(60.0, "Hz"),
)

## 4.3 Time-Domain Results

### Electromagnetic Torque

In [ ]:
results4.plot_torque().show()

### Rotor Speed

The speed ramps up and, thanks to the closed speed loop, remains close to the synchronous speed set by `frequency_ref` (nominal frequency) even after the load torque step at t = 1.5 s -- contrasting with the open-loop V/f case (tutorial_part_7.ipynb), where the nominal load introduces speed droop through slip.

In [ ]:
results4.plot_speed().show()

### Stator Phase Currents

In [ ]:
results4.plot_phase_currents(reference_frame="a-b-c").show()

### Stator Phase Voltages

As with `InverterVF`, the voltage waveforms show the characteristic SVPWM pattern, now synthesized from the FOC voltage references instead of a fixed V/f law.

In [ ]:
results4.plot_phase_voltages().show()

### Stator Line Voltages

In [ ]:
results4.plot_line_voltages().show()

## 4.4 Frequency-Domain Results (FFT)

As in tutorial_part_7.ipynb (Section 3.4), every FFT figure below restricts the displayed band to **0.5 Hz - 2.1 x Fs** via `frequency_range`.

### FFT of the Electromagnetic Torque

In [ ]:
results4.plot_torque(
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

### FFT of the Stator Phase Currents

In [ ]:
results4.plot_phase_currents(
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

## 4.5 Reference Frames - Park (d-q)

In the rotor-flux reference frame used internally by `InverterFOC`, the d-axis current tracks the (approximately constant) flux-producing reference current `ids_ref`, while the q-axis current is proportional to the electromagnetic torque. Both are visible below through the Park (d-q) reference frame.

In [ ]:
results4.plot_phase_currents(reference_frame="d-q").show()

In [ ]:
results4.plot_phase_currents(
    reference_frame="d-q",
    domain="frequency",
    frequency_range=Q_((0.5, 2.1 * Fs), "Hz"),
).show()

## 4.6 Comparing V/f and FOC Speed Control

To compare against the `InverterVF` drive from tutorial_part_7.ipynb, that simulation is repeated below (same motor, same reference of 30 Hz) so that this notebook remains self-contained.

In [ ]:
motor3 = rs.MotorElement(
    n=0,
    tag="TPIM_VF",
    power_nom=Q_(1.5, "hp"),
    voltage_nom=127,
    speed_nom=Q_(1710, "RPM"),
    frequency_nom=Q_(60.0, "Hz"),
    n_poles=4,
    stator_resistance=2.5,
    rotor_resistance=1.8,
    stator_reactance=1.3,
    rotor_reactance=1.3,
    mutual_reactance=43.08,
    Ip_motor=0.0372,
    viscosity_coeff=0.0,
    Ip_load=0.0,
)

t3 = t4  # same time grid used in Section 4

results3 = motor3.run_with_inverter_vf(
    t3,
    time_step=1e-5,
    load_torque_entrance_time=1.5,
    load_torque_ratio=1.0,
    frequency_s=Q_(Fs, "Hz"),
    time_ramp=0.6667,
    frequency_ref=Q_(30.0, "Hz"),
)

The plot below overlays the shaft speed obtained with `InverterVF` (`frequency_ref` = 30 Hz, roughly half of the nominal frequency) and with `InverterFOC` (this section, `frequency_ref` = 60 Hz, the nominal frequency). Beyond tracking a different reference, the key qualitative difference is how each responds to the load torque step at t = 1.5 s: the open-loop V/f drive settles at a lower speed set by the induction motor's natural slip, while the closed-loop FOC drive is actively regulated back towards its speed reference.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results3.t, y=results3.speed * 60 / (2 * np.pi),
    name="InverterVF (frequency_ref = 30 Hz)",
))
fig.add_trace(go.Scatter(
    x=results4.t, y=results4.speed * 60 / (2 * np.pi),
    name="InverterFOC (frequency_ref = 60 Hz)",
))
fig.update_layout(
    title="Shaft Speed: V/f vs. FOC",
    xaxis_title="Time (s)",
    yaxis_title="Speed (RPM)",
)
fig.show()

## References

- Wu, B. & Narimani, M. (2016). *High-Power Converters and AC Drives*. Wiley.
- Novotny, D. & Lipo, T. (1996). *Vector Control and Dynamics of AC Drives*. Oxford University Press.